In [1]:
import numpy as np
import pandas as pd
import pickle


In [2]:
df = pd.read_csv("../data/sentiment_reviews.csv")


In [3]:
product_text = pd.read_csv("../data/product_text.csv")


In [4]:
cosine_sim = np.load("../models/cosine_sim.npy")


In [5]:
P = np.load("../models/P.npy")
Q = np.load("../models/Q.npy")


In [6]:
with open("../models/user_map.pkl", "rb") as f:
    user_map = pickle.load(f)

with open("../models/product_map.pkl", "rb") as f:
    product_map = pickle.load(f)


In [7]:
def get_cf_score(user_id, product_id):
    try:
        # find encoded indices
        u_idx = df[df["UserId"] == user_id]["user_index"].iloc[0]
        p_idx = df[df["ProductId"] == product_id]["product_index"].iloc[0]
        
        return np.dot(P[u_idx], Q[p_idx])
    except:
        return None


In [8]:
indices = pd.Series(product_text.index, index=product_text["ProductId"])


In [9]:
def get_content_similarity(product_id1, product_id2):
    if product_id1 not in indices or product_id2 not in indices:
        return 0

    idx1 = indices[product_id1]
    idx2 = indices[product_id2]

    return cosine_sim[idx1][idx2]


In [10]:
sentiment_map = df.groupby("ProductId")["avg_sentiment_score"].mean().to_dict()


In [11]:
def get_sentiment_score(product_id):
    return sentiment_map.get(product_id, 0.5)


In [12]:
def hybrid_score(user_id, product_id, reference_product=None):
    
    cf_score = get_cf_score(user_id, product_id)
    sentiment_score = get_sentiment_score(product_id)
    
    # content similarity uses a reference product such as last viewed item
    content_score = 0
    if reference_product:
        content_score = get_content_similarity(reference_product, product_id)
    
    # hybrid formula
    final = (
        0.6 * (cf_score if cf_score is not None else 0)
        + 0.2 * content_score
        + 0.2 * sentiment_score
    )
    
    return final


In [13]:
def recommend_for_user(user_id, reference_product=None, top_n=10):
    
    # get all products
    all_products = df["ProductId"].unique()
    
    scores = []
    for p in all_products:
        score = hybrid_score(user_id, p, reference_product)
        scores.append((p, score))
    
    # sort by score
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    
    return scores[:top_n]


In [14]:
sample_user = df["UserId"].iloc[0]
sample_product = df["ProductId"].iloc[0]

recommend_for_user(sample_user, sample_product)


[('B001E4KFG0', 0.2),
 ('B000LQOCH0', 0.2),
 ('B006K2ZZ7K', 0.2),
 ('B000E7L2R4', 0.2),
 ('B00171APVA', 0.2),
 ('B0001PB9FE', 0.2),
 ('B001GVISJM', 0.2),
 ('B00144C10S', 0.2),
 ('B0001PB9FY', 0.2),
 ('B003F6UO7K', 0.2)]

In [15]:
with open("../models/hybrid_model.pkl", "wb") as f:
    pickle.dump({
        "P": P,
        "Q": Q,
        "indices": indices.to_dict(),
        "sentiment_map": sentiment_map,
        "product_text": product_text,
        "cosine_sim": cosine_sim
    }, f)
